<!-- beginner-banner-v1 -->

> 🧭 **비개발자 수강생 안내** — 이 노트북에서 새로 배우는 것: 본인 프로젝트 DB 에 Vanna 붙이고 **In-Chat Training** 으로 오답을 정답으로 가르치기.
>
> - 📖 강의 페이지: [day3/14-vanna-training](https://siapapa.github.io/day3/14-vanna-training/)
> - 🆕 처음이라면 → [비개발자 학습 가이드](https://siapapa.github.io/beginners-guide/)
> - 🔤 모르는 단어 → [용어 사전](https://siapapa.github.io/appendix/glossary/)
> - 🛠️ 환경/접속 막힘 → [사전 준비](https://siapapa.github.io/setup/) · [트러블슈팅](https://siapapa.github.io/appendix/troubleshooting/)
>
> **셀은 위에서 아래로 차례대로 실행**하세요. 시연용 코드(`구경만 하세요` 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 11. Vanna 자가학습 — DDL · Documentation · SQL Pairs
> Day 3 · 14H · 소요 약 50분

## 학습 목표

- **DDL · Documentation · SQL Pairs** 3 종 학습 자산을 `vn.train(...)` 으로 주입한다.
- 학습 **전 / 후** 동일 질문셋의 정답률을 측정해 학습 효과를 정량 비교한다.
- **In-Chat Training** 으로 오답을 정답 SQL 로 교정하고 재테스트한다.
- 본인 프로젝트 DB 에 동일한 학습 절차를 적용하는 가이드를 익힌다.

> **공유 DB 로 먼저 흐름을 익히고, 프로젝트 트랙에서는 본인 DB 로 교체하세요.** 이 노트북은 수업 일관성을 위해 병원 DB 를 사용합니다. 17 번(SQL 에이전트 빌드) 에서 본인 프로젝트에 Vanna 를 결합하는 흐름이 이어집니다.


In [ ]:
%pip install -q "vanna[chromadb,openai,postgres]" chromadb openai sqlalchemy psycopg2-binary pandas

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os

def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

# Vanna 는 LLM(=OpenAI) + DB(=Neon) 둘 다 직접 접근하므로 두 키 모두 필수.
_load_secret("NEON_DSN", required=True)
_load_secret("OPENAI_API_KEY", required=True)

print("Environment ready.")

## 1. Vanna 인스턴스 재생성 — self-contained

10 번 노트북과 동일한 클래스·퍼시스트 경로(`./vanna_chroma`) 를 사용합니다. 같은 런타임이면 10 번에서 만든 컬렉션을 이어서 사용하고, 런타임이 새로 시작되었다면 이 노트북이 처음부터 학습을 쌓습니다.


In [ ]:
# Vanna 인스턴스 만들기 — Vanna 는 "벡터 저장소 + LLM" 을 다중 상속으로 합쳐 쓰는 구조입니다.
# - ChromaDB_VectorStore : 학습 자산(DDL/Docs/SQL Pairs)을 벡터로 저장·검색.
# - OpenAI_Chat          : 질문 + 검색된 학습 자산을 받아 SQL 을 생성.
# 다중 상속이라 두 부모 클래스를 동시에 __init__ 해야 합니다 (아래 MyVanna).
from vanna.openai import OpenAI_Chat
from vanna.chromadb import ChromaDB_VectorStore
from urllib.parse import urlparse


class MyVanna(ChromaDB_VectorStore, OpenAI_Chat):
    def __init__(self, config=None):
        # 두 부모 클래스를 명시적으로 초기화 — Python 의 다중 상속 관용 패턴.
        ChromaDB_VectorStore.__init__(self, config=config)
        OpenAI_Chat.__init__(self, config=config)


# 설정값 한 dict 에 OpenAI 키, 모델, Chroma 영속 경로를 모아 전달.
vn = MyVanna(config={
    "api_key": os.environ["OPENAI_API_KEY"],
    "model": "gpt-4o-mini",
    "path": "./vanna_chroma",   # 디스크 영속 경로 — 같은 path 로 다시 열면 학습 자산이 그대로 살아난다.
})


def _parse_dsn(dsn: str) -> dict:
    """SQLAlchemy DSN 문자열을 Vanna 가 원하는 호스트/DB/유저/비번/포트 dict 로 분해."""
    u = urlparse(dsn)
    return {
        "host": u.hostname,
        "dbname": (u.path or "/").lstrip("/").split("?")[0],  # ?sslmode 같은 쿼리 제거
        "user": u.username,
        "password": u.password,
        "port": u.port or 5432,   # 기본 PostgreSQL 포트
    }


# Vanna 는 자체적으로 psycopg2 커넥션을 갖고 SQL 을 직접 실행합니다.
vn.connect_to_postgres(**_parse_dsn(os.environ["NEON_DSN"]))
print("Vanna ready. Connected to Neon.")

## 2. 학습 자산 ① — DDL 주입

각 `CREATE TABLE` 문을 **컬럼 주석(`-- ...`)** 과 함께 넘기면 Vanna 가 이를 ChromaDB 의 DDL 컬렉션에 저장합니다. 질문이 들어올 때 벡터 유사도로 관련 테이블만 검색해 프롬프트에 싣습니다.


In [ ]:
# 학습 자산 ① — DDL 주입.
# 각 CREATE TABLE 문을 vn.train(ddl=...) 로 넘기면 Vanna 가 ChromaDB 의 DDL 컬렉션에 임베딩으로 저장합니다.
# 질문이 들어오면 (질문 ↔ DDL) 의미 유사도로 관련 테이블만 끌어와 프롬프트에 싣습니다.
ddl_statements = [
    """
    CREATE TABLE departments (
        department_id   SERIAL PRIMARY KEY,
        name            VARCHAR(50) NOT NULL,
        floor           INT,
        phone           VARCHAR(20)
    );
    -- departments: 병원의 진료과 정보 (내과, 외과, 소아과 등)
    """,
    """
    CREATE TABLE doctors (
        doctor_id       SERIAL PRIMARY KEY,
        name            VARCHAR(100) NOT NULL,
        department_id   INT NOT NULL REFERENCES departments(department_id),
        specialty       VARCHAR(100),
        hire_date       DATE NOT NULL,
        salary          NUMERIC(12,2)
    );
    -- doctors: 의사 정보. department_id 로 진료과 참조.
    """,
    """
    CREATE TABLE patients (
        patient_id      SERIAL PRIMARY KEY,
        name            VARCHAR(100) NOT NULL,
        birth_date      DATE NOT NULL,
        gender          CHAR(1) NOT NULL CHECK (gender IN ('M','F')),
        phone           VARCHAR(20),
        address         VARCHAR(200),
        blood_type      VARCHAR(3) CHECK (blood_type IN ('A','B','O','AB')),
        created_at      TIMESTAMP DEFAULT NOW()
    );
    -- patients: 환자 기본 정보. gender 는 M(남) / F(여).
    """,
    """
    CREATE TABLE visits (
        visit_id        SERIAL PRIMARY KEY,
        patient_id      INT NOT NULL REFERENCES patients(patient_id),
        doctor_id       INT NOT NULL REFERENCES doctors(doctor_id),
        visit_date      DATE NOT NULL,
        visit_type      VARCHAR(20) NOT NULL CHECK (visit_type IN ('outpatient','inpatient','emergency')),
        status          VARCHAR(20) NOT NULL DEFAULT 'scheduled'
                        CHECK (status IN ('scheduled','completed','cancelled','no_show')),
        chief_complaint TEXT,
        cost            NUMERIC(10,2)
    );
    -- visits: 진료 방문 기록. visit_type 은 외래/입원/응급. status='completed' 만 유효 진료로 집계.
    """,
    """
    CREATE TABLE diagnoses (
        diagnosis_id    SERIAL PRIMARY KEY,
        visit_id        INT NOT NULL REFERENCES visits(visit_id),
        icd_code        VARCHAR(10) NOT NULL,
        description     VARCHAR(200) NOT NULL,
        severity        VARCHAR(10) CHECK (severity IN ('mild','moderate','severe'))
    );
    -- diagnoses: 진단 기록. severity 는 mild(경증)/moderate(중등)/severe(중증).
    """,
]

for ddl in ddl_statements:
    vn.train(ddl=ddl)
    # 학습된 DDL 의 테이블명을 추출해 진행 로그를 깔끔하게 출력 — "CREATE TABLE 다음 ~ 첫 ( 사이" 구간.
    table_name = ddl.split("CREATE TABLE")[1].split("(")[0].strip()
    print(f"  trained DDL: {table_name}")

print(f"\nDDL training complete: {len(ddl_statements)} tables.")

In [ ]:
# Check training corpus after DDL.
td = vn.get_training_data()
if td is None:
    print("Training data still empty (unexpected).")
else:
    print(f"Total training rows: {len(td)}")
    if "training_data_type" in td.columns:
        print(td["training_data_type"].value_counts())


## 3. 학습 자산 ② — Documentation (비즈니스 용어집)

DDL 만으로는 전달되지 않는 **도메인 규칙**을 자연어로 학습시킵니다. 예: "`status='completed'` 만 유효한 진료", "'지난달' = `DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1 month')`".


In [ ]:
# 학습 자산 ② — Documentation (자연어 비즈니스 규칙).
# DDL 만으로는 LLM 이 모를 도메인 규칙을 사람 말로 적어 학습시킵니다.
# 예: status 값의 의미, 나이 계산 공식, "지난달" 같은 시간 표현의 정의.
docs = [
    "visits 테이블에서 status='completed' 인 것만 실제 완료된 진료입니다. 'cancelled' 과 'no_show' 는 집계에서 제외합니다.",
    "visits.visit_type 의 값: 'outpatient' 은 외래, 'inpatient' 은 입원, 'emergency' 는 응급입니다.",
    "visits.cost 는 진료비(원, KRW). NULL 이면 아직 청구되지 않은 것입니다.",
    "환자의 나이를 계산하려면 EXTRACT(YEAR FROM AGE(birth_date)) 를 사용합니다.",
    "patients.gender 는 'M' 이 남성, 'F' 가 여성입니다.",
    "patients.blood_type 은 'A','B','O','AB' 중 하나입니다.",
    "diagnoses.severity 는 'mild'(경증) / 'moderate'(중등) / 'severe'(중증) 세 단계입니다.",
    "'재방문 환자' 란 같은 patient_id 로 visits 에 status='completed' 레코드가 2 건 이상 있는 환자입니다.",
    "'지난달' 이란 현재 월의 바로 전 달을 의미합니다. DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1 month') 이상이고 DATE_TRUNC('month', CURRENT_DATE) 미만.",
    "departments 는 병원의 진료과. 의사는 정확히 하나의 진료과에 소속됩니다.",
]

# 한 줄씩 vn.train(documentation=...) 로 주입 — 내부적으로 임베딩되어 Documentation 컬렉션에 저장됩니다.
for doc in docs:
    vn.train(documentation=doc)

print(f"Documentation training complete: {len(docs)} notes.")

## 4. 학습 자산 ③ — SQL Pairs (질문 · 정답 SQL)

**(질문, SQL)** 쌍은 Vanna 가 가장 강하게 활용하는 신호입니다. 질문이 들어오면 유사한 과거 질문의 SQL 을 Few-shot 으로 함께 실어 줍니다. 빈도 높은 질문부터 5~10 쌍 쌓는 것을 목표로 하세요.


In [ ]:
# 학습 자산 ③ — SQL Pairs (질문 ↔ 정답 SQL 쌍).
# Vanna 가 가장 강하게 활용하는 신호. 새 질문이 들어오면 의미가 비슷한 과거 질문의 SQL 을
# Few-shot 예시로 함께 LLM 에 주입 → 비슷한 패턴의 SQL 을 더 정확히 만들어 냅니다.
sql_pairs = [
    {
        "question": "전체 환자 수는 몇 명인가요?",
        "sql": "SELECT COUNT(*) AS total_patients FROM patients;",
    },
    {
        "question": "남성 환자 수는?",
        "sql": "SELECT COUNT(*) AS male_patients FROM patients WHERE gender = 'M';",
    },
    {
        "question": "진료과별 의사 수를 보여주세요.",
        "sql": """
            SELECT d.name AS department, COUNT(*) AS doctor_count
            FROM doctors doc
            JOIN departments d ON d.department_id = doc.department_id
            GROUP BY d.name
            ORDER BY doctor_count DESC;
        """,
    },
    {
        "question": "지난달 완료된 진료 건수는?",
        "sql": """
            SELECT COUNT(*) AS completed_visits
            FROM visits
            WHERE status = 'completed'
              AND visit_date >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1 month')
              AND visit_date <  DATE_TRUNC('month', CURRENT_DATE);
        """,
    },
    {
        "question": "가장 많이 방문한 환자 Top 5 는?",
        "sql": """
            SELECT p.name, COUNT(*) AS visit_count
            FROM visits v
            JOIN patients p ON p.patient_id = v.patient_id
            WHERE v.status = 'completed'
            GROUP BY p.patient_id, p.name
            ORDER BY visit_count DESC
            LIMIT 5;
        """,
    },
    {
        "question": "진료과별 평균 진료비를 보여줘",
        "sql": """
            SELECT dept.name AS department, ROUND(AVG(v.cost), 0) AS avg_cost
            FROM visits v
            JOIN doctors d ON d.doctor_id = v.doctor_id
            JOIN departments dept ON dept.department_id = d.department_id
            WHERE v.status = 'completed' AND v.cost IS NOT NULL
            GROUP BY dept.name
            ORDER BY avg_cost DESC;
        """,
    },
]

# vn.train(question=..., sql=...) 형식으로 한 쌍씩 주입.
for pair in sql_pairs:
    vn.train(question=pair["question"], sql=pair["sql"])

print(f"SQL pairs training complete: {len(sql_pairs)} pairs.")

In [ ]:
# Sanity check — training corpus summary.
td = vn.get_training_data()
if td is not None:
    print(f"Total rows: {len(td)}")
    if "training_data_type" in td.columns:
        print(td["training_data_type"].value_counts())


## 5. 학습 후 정확도 측정

10 번 노트북에서 실패했던 질문을 포함해 10 개 질문을 **일괄 실행**합니다. SQL 이 생성되고 Neon 에서 결과가 리턴되면 `ok`, 에러거나 빈 결과면 `fail` 로 기록합니다.


In [ ]:
# 학습 후 정확도 측정 — 10 개 질문을 일괄 실행해 ok / empty / err 로 분류한다.
import pandas as pd

# (질문, 기대 결과 형태) 튜플 리스트. 다양한 난이도와 형식이 섞여 있어야 학습 효과를 가늠할 수 있습니다.
test_questions = [
    ("전체 환자 수는?", "단일 숫자"),
    ("남성 환자 중 40세 이상은 몇 명?", "단일 숫자"),
    ("진료과별 의사 수를 보여줘", "진료과-의사수 표"),
    ("지난달 완료 진료 건수는?", "단일 숫자"),
    ("응급 진료 평균 비용은?", "단일 숫자"),
    ("가장 많이 방문한 환자 Top 3 는?", "환자명-방문수 표"),
    ("중증 진단을 받은 환자 이름은?", "환자명 목록"),
    ("2026년 월별 방문 수 추이는?", "월-방문수 표"),
    ("내과 의사 중 급여가 가장 높은 사람은?", "의사명+급여"),
    ("혈액형별 환자 분포는?", "혈액형-환자수 표"),
]

records = []
for question, expected in test_questions:
    try:
        # vn.generate_sql(질문) 은 학습 자산을 검색 → LLM 호출 → SQL 문자열 반환.
        sql = vn.generate_sql(question)
        # vn.run_sql(sql) 은 위에서 connect_to_postgres 로 잡아 둔 커넥션으로 실제 실행.
        df = vn.run_sql(sql)
        ok = df is not None and len(df) > 0
        records.append({
            "question": question,
            "expected": expected,
            "status": "ok" if ok else "empty",     # 비어 있는 결과는 ok 가 아님
            "rows": 0 if df is None else len(df),
            # 결과 표를 옆으로 길게 안 만들기 위해 줄바꿈 제거 + 90자 자르기.
            "sql_preview": (sql or "").replace("\n", " ")[:90],
        })
    except Exception as e:
        # SQL 생성 실패 또는 실행 에러는 'err:타입이름' 으로 분류.
        records.append({
            "question": question,
            "expected": expected,
            "status": f"err:{type(e).__name__}",
            "rows": 0,
            "sql_preview": str(e)[:90],
        })

df_results = pd.DataFrame(records)
print(df_results[["question", "status", "rows"]].to_string(index=False))
ok_count = int((df_results["status"] == "ok").sum())
print(f"\nPass rate: {ok_count}/{len(test_questions)} ({ok_count/len(test_questions)*100:.0f}%)")

## 6. In-Chat Training — 오답을 정답으로 교정

위 테스트에서 `empty` 또는 `err:` 로 끝난 질문에 대해 **올바른 SQL 을 수동 작성**해 다시 `vn.train(question=..., sql=...)` 으로 주입합니다. 동일 질문을 재실행해 정답으로 바뀌는지 확인합니다 — 이것이 Vanna 의 **피드백 루프**입니다.


In [ ]:
# Take up to 2 failed questions and patch with hand-written SQL.
failed = df_results[df_results["status"] != "ok"]
print(f"Failed this run: {len(failed)}")
for _, row in failed.iterrows():
    print(" -", row["question"], "|", row["sql_preview"][:60])


In [ ]:
# In-Chat Training — "이 질문은 이 SQL 이 정답이야" 라고 직접 가르치는 단계.
# 위 테스트에서 실패한 질문 2개에 손으로 작성한 정답 SQL 을 매핑해 학습.
corrected_pairs = [
    {
        "question": "중증 진단을 받은 환자 이름은?",
        "sql": """
            SELECT DISTINCT p.name
            FROM patients p
            JOIN visits v ON v.patient_id = p.patient_id
            JOIN diagnoses dg ON dg.visit_id = v.visit_id
            WHERE dg.severity = 'severe';
        """,
    },
    {
        "question": "2026년 월별 방문 수 추이는?",
        "sql": """
            SELECT TO_CHAR(visit_date, 'YYYY-MM') AS month,
                   COUNT(*) AS visit_count
            FROM visits
            WHERE EXTRACT(YEAR FROM visit_date) = 2026
              AND status = 'completed'
            GROUP BY TO_CHAR(visit_date, 'YYYY-MM')
            ORDER BY month;
        """,
    },
]

# 핵심: 같은 질문을 다시 학습시키면 기존 잘못된 패턴보다 새 SQL 이 우선 적용됩니다.
for pair in corrected_pairs:
    vn.train(question=pair["question"], sql=pair["sql"])
    print(f"  corrected: {pair['question']}")

# 같은 질문을 재실행 — 학습이 즉시 반영되는지 확인.
print("\nRe-test after correction:")
for pair in corrected_pairs:
    try:
        sql = vn.generate_sql(pair["question"])
        df = vn.run_sql(sql)
        # rows 가 0보다 크면 정답 SQL 로 잘 바뀌었다는 신호.
        print(f"  Q: {pair['question']} -> rows={0 if df is None else len(df)}")
    except Exception as e:
        print(f"  Q: {pair['question']} -> err:{type(e).__name__}")

## 7. 학습 자산 백업

`./vanna_chroma` 는 Colab 런타임이 종료되면 사라질 수 있습니다. 학습 내용을 CSV 로 덤프해 두면 다음 세션에서 빠르게 복원할 수 있습니다.


In [ ]:
td = vn.get_training_data()
if td is not None and len(td) > 0:
    td.to_csv("vanna_training_backup.csv", index=False)
    print(f"Backup written: vanna_training_backup.csv ({len(td)} rows)")
else:
    print("No training data to back up.")


## 8. 본인 프로젝트 DB 에 Vanna 적용 — 가이드

수업 일관성을 위해 위 절차는 병원 DB 로 돌렸습니다. **프로젝트 트랙에서는 본인 DB 로 교체하세요.**

1. Colab Secrets 에 `MY_PROJECT_DSN` 등록 후 `_parse_dsn(...)` 의 입력을 교체.
2. 본인 스키마의 `CREATE TABLE` 문 전부를 `vn.train(ddl=...)` 로 주입.
3. 비즈니스 용어 최소 5 개를 `vn.train(documentation=...)` 로 학습.
4. (질문, 정답 SQL) 쌍을 Easy 2 + Medium 2 이상 학습.
5. 10 개 테스트 질문으로 정확도 측정 → 실패 항목 교정(In-Chat Training).

> **목표:** 학습 전 대비 학습 후 정답률이 얼마나 개선되는지 **숫자로 기록**해 프로젝트 발표에 싣습니다.


In [ ]:
# TODO: 본인 프로젝트 DSN 으로 교체한 뒤 동일 절차를 반복하세요.
# _load_secret("MY_PROJECT_DSN", required=True)
# my_vn = MyVanna(config={
#     "api_key": os.environ["OPENAI_API_KEY"],
#     "model": "gpt-4o-mini",
#     "path": "./my_project_vanna_chroma",
# })
# my_vn.connect_to_postgres(**_parse_dsn(os.environ["MY_PROJECT_DSN"]))
#
# my_ddl = ["CREATE TABLE ... -- 주석", ...]
# for d in my_ddl:
#     my_vn.train(ddl=d)
#
# my_docs = [
#     "어떤 컬럼은 무엇을 의미합니다...",
# ]
# for doc in my_docs:
#     my_vn.train(documentation=doc)
#
# my_pairs = [
#     {"question": "...", "sql": "SELECT ..."},
# ]
# for p in my_pairs:
#     my_vn.train(question=p["question"], sql=p["sql"])
#
# # 측정:
# # my_vn.generate_sql("본인 도메인 질문")


## 실습 과제

다음 3 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. 학습 전후 정답률 비교
13H에서 측정한 베이스라인 정답률과 비교하세요!

아래 10개 질문을 `vn.generate_sql()` + `vn.run_sql()` 로 돌려 학습 후 정답률을 측정하세요.

| # | 질문 | 기대 형식 |
|---|---|---|
| 1 | 전체 환자 수는? | 단일 숫자 |
| 2 | 남성 환자 중 40세 이상은 몇 명? | 단일 숫자 |
| 3 | 진료과별 의사 수를 보여줘 | 진료과-의사수 표 |
| 4 | 지난달 완료 진료 건수는? | 단일 숫자 |
| 5 | 응급 진료 평균 비용은? | 단일 숫자 |
| 6 | 가장 많이 방문한 환자 Top 3는? | 환자명-방문수 표 |
| 7 | 중증 진단을 받은 환자 이름은? | 환자명 목록 |
| 8 | 2026년 월별 방문 수 추이는? | 월-방문수 표 |
| 9 | 내과 의사 중 급여가 가장 높은 사람은? | 의사명+급여 |
| 10 | 혈액형별 환자 분포는? | 혈액형-환자수 표 |

_힌트: 13H 베이스라인 측정 코드를 재사용하여 결과를 `pd.DataFrame` 으로 정리하고, `✅ / ⚠️ / ❌` 카운트로 정답률을 출력하세요._

### 2. 학습 전후 정답률 비교표 작성
아래 표를 복사하여 본인의 결과를 채워 넣으세요. (코딩 과제가 아닌 기록·정리 과제입니다.)

| # | 질문 | 학습 전 | 학습 후 | 개선 |
|---|---|---|---|---|
| 1 | 전체 환자 수는? | ❌ / ✅ | ❌ / ✅ | - / 개선 |
| 2 | 남성 환자 중 40세 이상은? | ❌ / ✅ | ❌ / ✅ | - / 개선 |
| 3 | 진료과별 의사 수를 보여줘 | ❌ / ✅ | ❌ / ✅ | - / 개선 |
| 4 | 지난달 완료 진료 건수는? | ❌ / ✅ | ❌ / ✅ | - / 개선 |
| 5 | 응급 진료 평균 비용은? | ❌ / ✅ | ❌ / ✅ | - / 개선 |
| 6 | 가장 많이 방문한 환자 Top 3는? | ❌ / ✅ | ❌ / ✅ | - / 개선 |
| 7 | 중증 진단을 받은 환자 이름은? | ❌ / ✅ | ❌ / ✅ | - / 개선 |
| 8 | 2026년 월별 방문 수 추이는? | ❌ / ✅ | ❌ / ✅ | - / 개선 |
| 9 | 내과 의사 중 급여 최고는? | ❌ / ✅ | ❌ / ✅ | - / 개선 |
| 10 | 혈액형별 환자 분포는? | ❌ / ✅ | ❌ / ✅ | - / 개선 |
| | **합계** | **/10** | **/10** | |

### 3. 본인 프로젝트 적용 체크리스트
아래 항목을 순서대로 완료하세요. (체크리스트 — 코드는 본인 프로젝트 DSN에 맞춰 작성)

- [ ] 본인 Neon DSN으로 Vanna 연결
- [ ] DDL 학습 (본인 테이블 전체)
- [ ] Documentation 학습 (비즈니스 규칙 5개 이상)
- [ ] SQL Pairs 학습 (최소 4개)
- [ ] 학습 전 5개 질문 테스트 (정답률 기록)
- [ ] 학습 후 10개 질문 테스트 (정답률 기록)
- [ ] 실패 질문 In-Chat Training
- [ ] 최종 정답률 비교


In [ ]:
# ============================================================
# 실습 과제 — 학습 전후 정답률 비교 / 비교표 / 본인 프로젝트 적용
# ============================================================

# 실습 1: 학습 전후 정답률 비교 (10개 질문)
# TODO: 13H 베이스라인 코드를 재사용해 위 10개 질문을 vn.generate_sql + vn.run_sql 로 돌리고 ✅ / ⚠️ / ❌ 카운트로 정답률을 출력하세요.
# 여기에 구현하세요.

# 실습 2: 학습 전후 정답률 비교표 작성
# TODO: 위 markdown 표(학습 전 / 학습 후 / 개선)를 본인 결과로 채워 별도 노트나 발표 자료에 정리하세요. (코딩 과제 아님)
# 여기에 구현하세요.

# 실습 3: 본인 프로젝트 적용 체크리스트
# TODO: 위 체크리스트 8개 항목을 본인 프로젝트 DSN으로 차례대로 수행하세요. (체크리스트 — 본인 프로젝트 코드)
# 여기에 구현하세요.


## 다음 노트북에서는…

Vanna 는 "RAG + LLM" 을 **완성된 도구**로 제공했지만, 실제 에이전트를 만들려면 **체인의 재료**를 직접 조합해야 합니다. 다음 **`12_langchain_lcel.ipynb`** 에서 `PromptTemplate | Model | Parser` 의 LCEL 파이프 연산자, `invoke` / `stream` / `batch`, 그리고 `with_structured_output` 을 익혀 Day 3 후반(LangGraph SQL 에이전트) 의 토대를 만듭니다.
